# AI Research Assistant System
Semantic search, summarization, keyword+entity extraction, RAG Q&A, topic clustering, and PDF-based paper matching over the ML-ArXiv-Papers dataset.

## 0. Setup

### 0.1 Install dependencies

In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu transformers==4.46.3 \
    keybert bertopic pypdf groq

### 0.2 Imports (all imports live here)

In [ ]:
import os
import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import faiss

from transformers import pipeline
from keybert import KeyBERT
from bertopic import BERTopic

from pypdf import PdfReader
from groq import Groq
from google.colab import userdata, files

from IPython.display import display, Markdown

### 0.3 (Optional) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Data Loading & Cleaning

In [ ]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
df = pd.DataFrame(dataset)
df = df[["title", "abstract"]]

# Reduce dataset size for faster iteration in Colab
df = df.head(15000)

print(df.shape)
df.isnull().sum()

Combine `title` + `abstract` into a single `paper_text` field used for embeddings.

In [ ]:
df["paper_text"] = (df["title"] + " " + df["abstract"]).str.replace("\n", "", regex=False).str.strip()
df[["paper_text"]].head()

## 2. Embeddings (Sentence Transformers)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

EMB_PATH = "/content/paper_embeddings.npy"
if os.path.exists(EMB_PATH):
    print("Loading cached embeddings")
    embeddings = np.load(EMB_PATH)
else:
    print("Encoding embeddings")
    embeddings = model.encode(df["paper_text"].to_list(), batch_size=32, show_progress_bar=True)
    np.save(EMB_PATH, embeddings)

print("Embeddings shape:", embeddings.shape)

## 3. FAISS Index (cosine similarity via normalized inner product)

In [ ]:
INDEX_PATH = "/content/paper_faiss.index"

if os.path.exists(INDEX_PATH):
    print("Loading existing FAISS index")
    index = faiss.read_index(INDEX_PATH)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

print("Total vectors indexed:", index.ntotal)

In [ ]:
def search_paper(query, top_k=5):
    """Semantic search over the paper index. Returns list of dicts."""
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            "score": float(score),
            "title": df.iloc[idx]["title"],
            "abstract": df.iloc[idx]["abstract"],
        })
    return results

for r in search_paper("deep learning for medical image analysis"):
    print(f"{r['score']:.3f}  {r['title']}")

## 4. Abstract Summarization (BART)

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

In [ ]:
def summarize_abstract(abstract, max_length=120, min_length=40):
    result = summarizer(abstract, max_length=max_length, min_length=min_length, do_sample=False)
    return result[0]["summary_text"]

sample = df.iloc[0]
print(summarize_abstract(sample["abstract"]))

## 5. Keyword Extraction (KeyBERT)

In [ ]:
kw_model = KeyBERT(model=model)

def extract_keywords(text, top_n=5):
    return kw_model.extract_keywords(
        text, keyphrase_ngram_range=(1, 3), stop_words="english", top_n=top_n
    )

extract_keywords(df.iloc[0]["abstract"])

## Feature 1: Keywords with Entity Type
Classifies each extracted keyword into a category (Programming Language, Framework, Model Architecture, Dataset, etc.) using a curated dictionary with a zero-shot classification fallback.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

CANDIDATE_LABELS = [
    "Programming Language", "Framework", "Library",
    "Algorithm", "Model Architecture", "Dataset",
    "Evaluation Metric", "Research Concept or Technique",
    "Application Domain", "Data Type or Modality",
    "Task or Problem Type", "Image Processing Task",
]

# Curated dictionary - expand as you find more common ML/CS terms
ENTITY_DICT = {
    "python": "Programming Language", "java": "Programming Language", "c++": "Programming Language",
    "fastapi": "Framework", "django": "Framework", "flask": "Framework", "react": "Framework",
    "pytorch": "Library", "tensorflow": "Library", "scikit-learn": "Library", "keras": "Library",
    "bert": "Model Architecture", "gpt": "Model Architecture", "resnet": "Model Architecture",
    "transformer": "Model Architecture", "lstm": "Model Architecture", "cnn": "Model Architecture",
    "cnns": "Model Architecture", "rnn": "Model Architecture", "alexnet": "Model Architecture",
    "convolutional neural network": "Model Architecture", "convolutional neural networks": "Model Architecture",
    "bleu": "Evaluation Metric", "f1 score": "Evaluation Metric", "accuracy": "Evaluation Metric",
    "precision": "Evaluation Metric", "recall": "Evaluation Metric",
    "imagenet": "Dataset", "coco": "Dataset", "mnist": "Dataset",
    "gradient descent": "Algorithm", "backpropagation": "Algorithm", "random forest": "Algorithm",
}

In [ ]:
def get_entity_type(keyword, confidence_threshold=0.25):
    keyword_clean = keyword.lower().strip()

    # Step 1: dictionary lookup (fast, precise)
    if keyword_clean in ENTITY_DICT:
        return ENTITY_DICT[keyword_clean], 1.0

    # Step 2: fallback to zero-shot classification
    result = zero_shot(keyword, CANDIDATE_LABELS, hypothesis_template="This term refers to a {}.")
    label, score = result["labels"][0], result["scores"][0]

    if score < confidence_threshold:
        return "Unclassified", score
    return label, score

test_keywords = ["python", "fastapi", "bert", "gradient boosting", "attention mechanism"]
for kw in test_keywords:
    entity_type, confidence = get_entity_type(kw)
    print(f"{kw:25s} -> {entity_type:25s} (confidence: {confidence:.2f})")

### Full pipeline: search + summarize + keywords + entity types

In [ ]:
def search_and_summarize(query, top_k=5):
    """Returns a list of dicts: score, title, abstract, summary, keywords (with entity types)."""
    results = search_paper(query, top_k=top_k)

    for r in results:
        r["summary"] = summarize_abstract(r["abstract"])

        keywords = extract_keywords(r["abstract"], top_n=5)
        r["keywords"] = [
            {"keyword": kw, "entity_type": get_entity_type(kw)[0], "confidence": round(get_entity_type(kw)[1], 2)}
            for kw, _ in keywords
        ]
    return results

results = search_and_summarize("deep learning for medical image analysis")
for r in results:
    print(r["title"])
    print("Summary:", r["summary"])
    for k in r["keywords"]:
        print(f"  - {k['keyword']} -> {k['entity_type']} ({k['confidence']})")
    print()

## Feature 2: RAG Chatbot
Retrieval-augmented Q&A over the paper index using Groq (`openai/gpt-oss-120b`).

In [ ]:
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

In [ ]:
def retrieve_context(query, top_k=5):
    return search_paper(query, top_k=top_k)

def ask_research_assistant(question, top_k=5, max_tokens=800):
    """Returns a dict: {question, answer, sources}."""
    contexts = retrieve_context(question, top_k=top_k)

    context_text = ""
    for i, c in enumerate(contexts):
        context_text += f"[Paper {i+1}] Title: {c['title']}\nAbstract: {c['abstract'][:600]}\n\n"

    system_prompt = (
        "You are a research assistant. Answer the user's question using ONLY "
        "the provided paper excerpts. Cite papers as [Paper 1], [Paper 2], etc. "
        "If the excerpts don't contain enough information, say so honestly."
    )
    user_prompt = f"Context:\n{context_text}\nQuestion: {question}\n\nAnswer:"

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=max_tokens,
    )

    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "sources": contexts,
    }

def display_answer(result):
    md_output = f"### 🔎 Question\n{result['question']}\n\n### 💡 Answer\n{result['answer']}\n\n### 📚 Sources\n"
    for i, c in enumerate(result["sources"]):
        md_output += f"{i+1}. **{c['title'].strip()}** (relevance: {c['score']:.3f})\n"
    display(Markdown(md_output))

In [ ]:
result = ask_research_assistant("What are the main challenges in using deep learning for medical imaging?")
display_answer(result)

## Feature 3: Topic Clustering
Auto-clusters papers into topics using BERTopic on top of the existing sentence-transformer embeddings.

In [ ]:
topic_model = BERTopic(embedding_model=model, verbose=True)
topics, probs = topic_model.fit_transform(df["paper_text"].to_list(), embeddings)

df["topic"] = topics
df["topic_label"] = df["topic"].map(
    lambda t: topic_model.get_topic_info().set_index("Topic")["Name"].get(t, "Outlier")
)

topic_model.get_topic_info().head(15)

In [ ]:
topic_model.visualize_topics()

In [ ]:
def papers_by_topic(topic_id, n=5):
    """Returns a list of dicts for papers under a given topic id."""
    rows = df[df["topic"] == topic_id][["title", "abstract"]].head(n)
    return rows.to_dict(orient="records")

papers_by_topic(0)

## Feature 4: PDF Upload -> Similar Papers
Extracts text from an uploaded PDF and finds the most similar papers in the index.

In [ ]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = " ".join(page.extract_text() or "" for page in reader.pages)
    return text.strip()

def find_similar_papers_from_pdf(pdf_path, top_k=5):
    pdf_text = extract_text_from_pdf(pdf_path)
    pdf_embedding = model.encode([pdf_text[:2000]])  # title+abstract length works best
    faiss.normalize_L2(pdf_embedding)
    D, I = index.search(pdf_embedding, top_k)

    return [
        {"title": df.iloc[idx]["title"], "score": float(score)}
        for score, idx in zip(D[0], I[0])
    ]

In [ ]:
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
find_similar_papers_from_pdf(pdf_filename)